# Kelvin — Paired Metamorphic Diagnostics for RAG

**Is your AI understanding your data — or guessing?**

Kelvin tests whether a RAG pipeline:
- **Stays stable** when irrelevant things change (reorder, pad)
- **Reacts correctly** when governing evidence changes (swap)

This notebook runs a complete Kelvin diagnostic on two venture-assessment cases using a local mock pipeline. No API key needed.

**What you'll see:**
1. Install Kelvin
2. Define a mock pipeline
3. Write two case files
4. Run `kelvin check`
5. Read the diagnostic report

→ [GitHub](https://github.com/SbenaVision/kelvin) · [Whitepaper](https://github.com/SbenaVision/kelvin/blob/main/docs/whitepaper.md)

## Step 1 — Install Kelvin

In [ ]:
!pip install kelvin -q

## Step 2 — Define a mock pipeline

Kelvin's `run:` command is any shell command that:
- Reads a markdown case file from `--input`
- Writes a JSON file with a scalar `decision_field` to `--output`

Here we use a mock pipeline that parses the `## Gate Rule` section and returns a deterministic `stage_assessment`. In a real project, this would call your actual RAG pipeline.

In [ ]:
pipeline_code = '''
import sys
import json
import argparse

def assess_stage(text):
    """Mock pipeline: reads gate rule, returns stage_assessment."""
    text_lower = text.lower()

    # Find the Gate Rule section
    gate_rule = ""
    in_gate = False
    for line in text.splitlines():
        if line.strip().lower().startswith("## gate rule"):
            in_gate = True
            continue
        if in_gate:
            if line.startswith("## "):
                break
            gate_rule += line + " "

    gate_lower = gate_rule.lower()

    # Deterministic decision based on gate rule content
    met_signals = [
        "all conditions are met",
        "conditions met",
        "committed $1m",
        "500+",
        "waitlist",
        "first ventures are onboarding",
    ]
    unmet_signals = [
        "none of these conditions",
        "not currently met",
        "no users",
        "no validation",
        "no committed",
    ]

    met_count = sum(1 for s in met_signals if s in gate_lower)
    unmet_count = sum(1 for s in unmet_signals if s in gate_lower)

    if met_count >= 2:
        stage = "seed"
    elif unmet_count >= 2:
        stage = "idea"
    else:
        stage = "pre-seed"

    return {"stage_assessment": stage}

parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
parser.add_argument("--output", required=True)
args = parser.parse_args()

with open(args.input) as f:
    text = f.read()

result = assess_stage(text)

with open(args.output, "w") as f:
    json.dump(result, f)
'''

with open("pipeline.py", "w") as f:
    f.write(pipeline_code)

print("pipeline.py written")

## Step 3 — Write two case files

Each case is a markdown file with `## Section` headers. Each header becomes a typed unit.

The `governing_types: [gate_rule]` in `kelvin.yaml` tells Kelvin which section to swap between cases to test sensitivity.

In [ ]:
import os
os.makedirs("cases", exist_ok=True)

# Case 1 — FreakingGenius: unmet gate conditions
case_fg = """## Venture Description
FreakinGenius is an AI-powered math tutoring service for middle school students.
The product combines a voice AI chatbot with a tablet app that analyzes handwritten
work in real-time. Subscription price: $79/month.

## Target Customer
Middle school students struggling with math. Families pay monthly.
90% receive a branded tablet via BNPL at $25-35/month over 36 months.

## Market Evidence
$12B K-12 tutoring market. Human tutors are expensive and scarce.
No existing product combines real-time handwriting analysis with voice AI.

## Traction Signal
Started building. No validation completed. No users. No paid customers.

## Team
Part-time founder investing up to $25K personal capital.
One full-time BSc graduate with math teaching experience.
One senior technical advisor at 5-10 hours per week.

## Gate Rule
Advance from Validate to Build requires: problem validated with paying or
committed families, evidence that voice-plus-handwriting experience is
meaningfully better than existing alternatives, and at least one distribution
channel with demonstrated conversion. None of these conditions are currently
met. No users, no validation, no committed customers.
"""

# Case 2 — Envelop: met gate conditions
case_env = """## Venture Description
Envelop is an AI operating partner that takes domain experts from business idea
to first revenue in weeks. The core product shows one Next Best Action at all
times. AI agents handle ~90% of execution tasks.

## Target Customer
Domain experts with a real business idea who lack conviction and a starting
point. Currently turning to ChatGPT for generic advice or hiring consultants.

## Market Evidence
Millions of domain experts spot opportunities but never act. No product
combines honest AI assessment with personalized execution in one platform.

## Traction Signal
Platform built and deployed. Five ventures onboarding. Assessment pipeline
running end-to-end. First ventures actively using the platform.

## Team
Two co-founders: one strategy/product, one senior engineer.
Bootstrap-only. No outside funding.

## Gate Rule
Advance from Validate to Build requires: founder committed capital, evidence
of demand, and first ventures actively using the platform. All conditions are
met. Founder has committed $1M to make it happen. A waitlist of 500+ potential
customers has been collected and validated. First ventures are onboarding now
with real usage beginning.
"""

with open("cases/freakinggenius.md", "w") as f:
    f.write(case_fg)

with open("cases/envelop.md", "w") as f:
    f.write(case_env)

print("cases/freakinggenius.md written")
print("cases/envelop.md written")

## Step 4 — Write kelvin.yaml

Four required keys:
- `run` — shell command to invoke your pipeline
- `cases` — folder of case files
- `decision_field` — the JSON key your pipeline writes (must be scalar)
- `governing_types` — unit types used for swap perturbations

In [ ]:
config = """run: python pipeline.py --input {input} --output {output}
cases: ./cases
decision_field: stage_assessment
governing_types: [gate_rule]
seed: 0
"""

with open("kelvin.yaml", "w") as f:
    f.write(config)

print("kelvin.yaml written")
print()
print(open("kelvin.yaml").read())

## Step 5 — Run Kelvin

Kelvin will:
1. Run a baseline for each case
2. Generate perturbations (reorder, pad, swap)
3. Run the pipeline on each perturbation
4. Score invariance and sensitivity
5. Print the diagnostic report

In [ ]:
!kelvin check

## Step 6 — Inspect the results

Kelvin writes full artifacts to disk. You can inspect every perturbation.

In [ ]:
import json

print("=== Global report ===")
with open("kelvin/report.json") as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

In [ ]:
print("=== FreakingGenius — what changed ===")
with open("kelvin/freakinggenius/report.json") as f:
    fg = json.load(f)

baseline = fg["baseline"]["decision_value"]
print(f"Baseline: {baseline}")
print()
for p in fg.get("perturbations", []):
    name     = p.get("variant_id", "?")
    decision = p["invocation"]["decision_value"]
    distance = p.get("distance") or 0
    changed  = " ← CHANGED" if distance > 0 else ""
    print(f"  {name}: {decision}{changed}")


In [ ]:
print("=== Envelop — what changed ===")
with open("kelvin/envelop/report.json") as f:
    env = json.load(f)

baseline = env["baseline"]["decision_value"]
print(f"Baseline: {baseline}")
print()
for p in env.get("perturbations", []):
    name     = p.get("variant_id", "?")
    decision = p["invocation"]["decision_value"]
    distance = p.get("distance") or 0
    changed  = " ← CHANGED" if distance > 0 else ""
    print(f"  {name}: {decision}{changed}")


## What's next

Replace `pipeline.py` with your actual RAG pipeline. The contract is simple:
- Read markdown from `--input`
- Write JSON with your `decision_field` to `--output`
- Exit non-zero on failure

Any pipeline callable from the command line works — Python, Node, Deno, bash, curl.

→ [GitHub](https://github.com/SbenaVision/kelvin)  
→ [Whitepaper](https://github.com/SbenaVision/kelvin/blob/main/docs/whitepaper.md)  
→ [README](https://github.com/SbenaVision/kelvin#readme)